# plot new twin plots

In [9]:
import sys
sys.path.append('/n/home06/drooryck/codeswitching-llms')
from pathlib import Path
import pandas as pd
import json
from typing import List

In [10]:
from importlib import reload
import july_aug_sept_exp.src.metrics as metrics
import july_aug_sept_exp.src.plotting as plotting

# Reload metrics first (since plotting might import metrics)
reload(metrics)
reload(plotting)

# Now get the classes
Metrics = metrics.Metrics
BilingualPlotter = plotting.BilingualPlotter

In [11]:
def collect_ablation_results(run_dirs: List[Path]) -> pd.DataFrame:
    results = []

    for run_dir in run_dirs:
        # Extract run parameters from directory name
        dir_name = run_dir.name
        if dir_name.startswith("p") and "_run" in dir_name:
            parts = dir_name.split("_run")
            try:
                # Handle decimal proportions by removing 'p' and converting directly to float
                prop = float(parts[0][1:]) / 100.0  # Just convert the string after 'p' to float
                run_id = int(parts[1])
            except ValueError:
                print(f"Couldn't parse proportion or run ID from {dir_name}")
                continue
        else:
            continue

        # Track if we found any metrics for this run
        found_metrics = False

        # Collect metrics for each ablation type
        for ablation_type in ['none', 'subject', 'verb', 'object']:
            metrics_file = run_dir / f"ablation_{ablation_type}_metrics.json"
            if not metrics_file.exists():
                continue

            try:
                with open(metrics_file, 'r') as f:
                    metrics = json.load(f)
                found_metrics = True
            except json.JSONDecodeError:
                print(f"Error reading metrics from {metrics_file}")
                continue

            result = {
                'prop': prop,
                'run_id': run_id,
                'run_dir': str(run_dir),
                'ablation': ablation_type,
                **metrics
            }
            results.append(result)

        if not found_metrics:
            print(f"No metrics found in {run_dir}")

    df = pd.DataFrame(results)
    if df.empty:
        print("Warning: No results collected!")
    else:
        print(f"Collected {len(df)} total results across {df['ablation'].nunique()} ablation types")
    
    return df


base_dir = Path("/n/home06/drooryck/codeswitching-llms/july_aug_sept_exp")  # Updated base directory
input_dir = base_dir / "results/sep24.3"  # Directory containing the input data
output_dir = base_dir / "scripts/temp_plots"  # New output directory for plots
output_dir.mkdir(parents=True, exist_ok=True)
run_dirs = list(input_dir.glob("p*_run*"))

# Collect all results
results_df = collect_ablation_results(run_dirs)

# Create plots for each ablation type
for ablation_type in ['none', 'subject', 'verb', 'object']:
    # Create plots directory for this ablation type
    plots_dir = output_dir / f"plots_{ablation_type}"
    plots_dir.mkdir(exist_ok=True)
    
    type_results = results_df[results_df['ablation'] == ablation_type]
    
    if not type_results.empty:
        print(f"\nCreating plots for {ablation_type} ablation")
        print(f"Found {len(type_results)} runs")
        
        # Create plotter with specific output directory for this ablation type
        plotter = BilingualPlotter(type_results, plots_dir)
        plotter.create_all_plots()
        print(f"Saved plots to {plots_dir}")
    else:
        print(f"No results found for {ablation_type} ablation")

print("\nDone!")

Collected 740 total results across 4 ablation types

Creating plots for none ablation
Found 185 runs
Saved plots to /n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/scripts/temp_plots/plots_none

Creating plots for subject ablation
Found 185 runs
Saved plots to /n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/scripts/temp_plots/plots_subject

Creating plots for verb ablation
Found 185 runs
Saved plots to /n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/scripts/temp_plots/plots_verb

Creating plots for object ablation
Found 185 runs
Saved plots to /n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/scripts/temp_plots/plots_object

Done!


In [ ]:
# Load a sample prediction to debug
metrics = Metrics(Path("/n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/data/lexicon_sep22.json"))

# Debug with some sample data
sample_pred = "de hond a mangé le loup"  # Example prediction
sample_input = "le chien aime le loup"  # Example input

print("Sample prediction:", sample_pred)
print("Tokenized prediction:", metrics.tokenize(sample_pred.lower()))
print("Structure check:", metrics.check_structure_conformity(sample_pred))

# Let's also check what tokens we get
tokens = metrics.tokenize(sample_pred.lower())
for i, token in enumerate(tokens):
    print(f"Token {i}: '{token}'")

Sample prediction: de hond a mangé le loup
Tokenized prediction: ['de', 'hond', 'a', 'mangé', 'le', 'loup']
Structure check: {'follows_fr_structure': True, 'follows_nl_structure': False, 'follows_either_structure': True, 'token_types': ['det_nl', 'noun_nl', 'aux_fr', 'part_fr', 'det_fr', 'noun_fr']}
Token 0: 'de'
Token 1: 'hond'
Token 2: 'a'
Token 3: 'mangé'
Token 4: 'le'
Token 5: 'loup'


In [ ]:
## takes long, use recompute_metrics.py parallel script.

# # Script to recompute metrics from existing predictions
# def recompute_metrics_for_run(run_dir: Path, lexicon_path: Path):
#     """Recompute metrics for a single run from existing predictions."""
#     predictions_file = run_dir / "ablation_predictions.csv"
#     if not predictions_file.exists():
#         print(f"No predictions file found in {run_dir}")
#         return
    
#     # Load predictions
#     pred_df = pd.read_csv(predictions_file)
    
#     # Initialize metrics with fixed code
#     metrics = Metrics(lexicon_path)
    
#     # Recompute metrics for each ablation type
#     for ablation_type in ["none", "subject", "verb", "object"]:
#         print(f"Recomputing {ablation_type} metrics for {run_dir.name}...")
#         type_preds = pred_df[pred_df["ablation"] == ablation_type].to_dict("records")
        
#         if type_preds:  # Only if we have predictions for this ablation type
#             new_metrics = metrics.compute_all_metrics(type_preds, ablation_type)
#             metrics.save_metrics(new_metrics, run_dir / f"ablation_{ablation_type}_metrics.json")

# # Usage
# lexicon_path = Path("/n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/data/lexicon_sep22.json")
# results_dir = Path("/n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/results/sep24.3")

# for run_dir in results_dir.glob("p*_run*"):
#     recompute_metrics_for_run(run_dir, lexicon_path)

Recomputing none metrics for p95.0_run5...
Recomputing subject metrics for p95.0_run5...
Recomputing verb metrics for p95.0_run5...
Recomputing object metrics for p95.0_run5...
Recomputing none metrics for p10.0_run2...
Recomputing subject metrics for p10.0_run2...


KeyboardInterrupt: 

In [ ]:
## see example sentences
import pandas as pd

df = pd.read_csv("/n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/results/sep24.3/p50.0_run5/ablation_predictions.csv")

subset = df[df['ablation'] == 'verb'].head(20)
subset = subset.drop('gold', axis=1)
subset

,language,input,prediction,ablation
2,fr,le chien mag le loup,le chien a gemogen le loup,verb
6,fr,les chiens mogen les loups,les chiens ont gemogen les loups,verb
10,fr,le chien slaat le loup,le chien a geslagen le loup,verb
14,fr,les chiens slaan les loups,les chiens ont geslagen les loups,verb
18,fr,le chien valt le loup,le chien a aangevallen le loup,verb
22,fr,les chiens vallen les loups,les chiens ont aangevallen les loups,verb
26,fr,le chien neemt le loup,le chien a genomen le loup,verb
30,fr,les chiens nemen les loups,les chiens ont genomen les loups,verb
34,fr,le chien trekt le loup,le chien a getrokken le loup,verb
38,fr,les chiens trekken les loups,les chiens ont getrokken les loups,verb
